# 실험 제목: 멀티턴 대화와 기억력 (Chat History)

**목표**: LLM API가 기본적으로 이전 대화를 기억하지 못함(Stateless)을 확인하고, `messages` 리스트에 대화 기록을 누적하여 문맥을 유지하는 방법을 실습합니다.

In [ ]:
# 필요한 라이브러리 import
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# .env 파일 로드 및 OpenAI 클라이언트 생성
load_dotenv()
client = OpenAI()

### 1. 기억력이 없는 단발성 호출 (Stateless)
API는 기본적으로 이전 요청을 기억하지 못합니다.

In [ ]:
# 첫 번째 질문
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕! 내 이름은 김철수야."}]
)
print("AI 답변 1:", response1.choices[0].message.content)

# 두 번째 질문 (단발성 호출로 이름 물어보기)
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "내 이름이 뭐라고 했지?"}]
)
print("AI 답변 2:", response2.choices[0].message.content)

### 2. Chat History를 포함한 멀티턴 대화
이전 대화 내역을 `messages` 배열에 계속 추가해서 보내야 문맥이 유지됩니다.

In [ ]:
# 대화 기록을 저장할 리스트 생성
chat_history = [
    {"role": "system", "content": "당신은 사용자의 질문에 친절하게 답하는 챗봇입니다."}
]

# 1. 사용자의 첫 번째 질문 추가
user_msg_1 = "안녕! 내 이름은 김철수야."
chat_history.append({"role": "user", "content": user_msg_1})

# API 호출 1
res1 = client.chat.completions.create(model="gpt-4o-mini", messages=chat_history)
ai_msg_1 = res1.choices[0].message.content
print("AI 답변 1:", ai_msg_1)

# 2. AI의 첫 번째 답변을 History에 추가 (매우 중요!)
chat_history.append({"role": "assistant", "content": ai_msg_1})

# 3. 사용자의 두 번째 질문 추가
user_msg_2 = "내 이름이 뭐라고 했지?"
chat_history.append({"role": "user", "content": user_msg_2})

# API 호출 2 (기록이 전부 담긴 chat_history 전달)
res2 = client.chat.completions.create(model="gpt-4o-mini", messages=chat_history)
ai_msg_2 = res2.choices[0].message.content
print("AI 답변 2:", ai_msg_2)

### 실험 결과 정리

* **Stateless 특성**: API를 매번 새로 호출하면 이전 질문을 기억하지 못합니다.
* **History 유지 방법**: `messages` 배열에 `user` 메시지와 `assistant` 메시지를 순차적으로 누적하여 전달해야 챗봇처럼 문맥을 이어갈 수 있음을 확인했습니다.

### Notion 실험 로그 기록용

* **결과**: `gpt-4o-mini` API의 Stateless 특성을 이해하고, `messages` 리스트 관리를 통해 멀티턴 대화(문맥 유지)를 성공적으로 구현함.
* **문제점**: 대화가 길어지면 `messages` 배열이 커져서 한 번에 보내는 토큰 양과 비용이 계속 증가함.
* **다음 개선 방향**: LangChain의 `Memory` 모듈을 사용하거나, 대화가 길어질 경우 이전 대화를 요약해서 넘기는 기법을 학습해 볼 예정.